In [ ]:
import pandas as pd

# Load the file - change path to where your file actually is
df = pd.read_csv('E:/Project/r4.2/logon.csv')

# 1. How many unique users
unique_users = df['user'].nunique()
print(f"Unique users: {unique_users}")

# 2. Average logons per user per day
df['date'] = pd.to_datetime(df['date'])
df['day'] = df['date'].dt.date

daily_logons = df.groupby(['user', 'day']).size().reset_index(name='count')
avg_logons = daily_logons['count'].mean()
print(f"Avg logons per user per day: {avg_logons:.2f}")

# 3. User with most logon events
top_user = df['user'].value_counts().idxmax()
top_count = df['user'].value_counts().max()
print(f"Most active user: {top_user} with {top_count} events")

In [ ]:
# After hours logons (before 7am or after 7pm)
df['hour'] = df['date'].dt.hour

after_hours = df[df['activity'] == 'Logon'].copy()
after_hours = after_hours[(after_hours['hour'] < 7) | (after_hours['hour'] >= 19)]

# Count after-hours logons per user
ah_counts = after_hours.groupby('user').size().reset_index(name='after_hours_logons')
ah_counts = ah_counts.sort_values('after_hours_logons', ascending=False)

print(ah_counts.head(10))

In [ ]:
# Load device and file logs
device_df = pd.read_csv('E:/Project/r4.2/device.csv')
file_df = pd.read_csv('E:/Project/r4.2/file.csv')

# Filter for WPR0368 only
target_user = 'WPR0368'

logon_count = df[df['user'] == target_user].shape[0]
device_count = device_df[device_df['user'] == target_user].shape[0]
file_count = file_df[file_df['user'] == target_user].shape[0]

print(f"Logon events: {logon_count}")
print(f"Device (USB) events: {device_count}")
print(f"File events: {file_count}")

In [ ]:
import pandas as pd

target_user = 'WPR0368'

# Only load the columns we need
http_df = pd.read_csv('E:/Project/r4.2/http.csv', usecols=['user'])
email_df = pd.read_csv('E:/Project/r4.2/email.csv', usecols=['from'])

http_count = http_df[http_df['user'] == target_user].shape[0]
email_count = email_df[email_df['from'] == target_user].shape[0]

print(f"HTTP events: {http_count}")
print(f"Email sent: {email_count}")

# Free memory immediately after
del http_df, email_df

In [ ]:
import pandas as pd

# Load only what we need
logon_df = pd.read_csv('E:/Project/r4.2/logon.csv')
device_df = pd.read_csv('E:/Project/r4.2/device.csv', usecols=['user'])
file_df = pd.read_csv('E:/Project/r4.2/file.csv', usecols=['user'])

# Parse datetime
logon_df['date'] = pd.to_datetime(logon_df['date'])
logon_df['hour'] = logon_df['date'].dt.hour

# Total logons per user
total_logons = logon_df.groupby('user').size().reset_index(name='total_logons')

# After hours logons per user
after_hours = logon_df[
    (logon_df['activity'] == 'Logon') &
    ((logon_df['hour'] < 7) | (logon_df['hour'] >= 19))
]
ah_counts = after_hours.groupby('user').size().reset_index(name='after_hours_logons')

# USB events per user
usb_counts = device_df.groupby('user').size().reset_index(name='usb_events')

# File events per user
file_counts = file_df.groupby('user').size().reset_index(name='file_events')

# Merge everything
profile = total_logons.merge(ah_counts, on='user', how='left')
profile = profile.merge(usb_counts, on='user', how='left')
profile = profile.merge(file_counts, on='user', how='left')
profile = profile.fillna(0)

print(profile.sort_values('after_hours_logons', ascending=False).head(10))

In [ ]:
# Simple risk scoring
profile['risk_score'] = (
    profile['total_logons'] * 0.2 +
    profile['after_hours_logons'] * 0.5 +
    profile['usb_events'] * 0.8 +
    profile['file_events'] * 0.7
)

# Normalise to 0-100
profile['risk_score'] = (
    (profile['risk_score'] - profile['risk_score'].min()) /
    (profile['risk_score'].max() - profile['risk_score'].min()) * 100
).round(2)

top_risks = profile.sort_values('risk_score', ascending=False).head(10)
print(top_risks[['user', 'total_logons', 'after_hours_logons', 'usb_events', 'file_events', 'risk_score']])

In [ ]:
import anthropic
import os
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

message = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "Say 'API connection successful' and nothing else."}
    ]
)

print(message.content[0].text)

In [ ]:
import anthropic
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

def generate_threat_report(user_row):
    prompt = f"""
You are a cybersecurity analyst reviewing insider threat indicators.

User: {user_row['user']}
Total Logon Events: {user_row['total_logons']}
After-Hours Logons: {user_row['after_hours_logons']}
USB/Device Events: {user_row['usb_events']}
File Access Events: {user_row['file_events']}
Risk Score: {user_row['risk_score']}/100

Based on these behavioural indicators, write a concise threat assessment. Include:
1. Threat level (Low / Medium / High / Critical)
2. Most suspicious behaviour pattern
3. Recommended action for the SOC team
Keep it under 100 words. Be direct and specific.
"""
    message = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

top3 = profile.sort_values('risk_score', ascending=False).head(3)

for _, row in top3.iterrows():
    print(f"\n{'='*50}")
    print(f"USER: {row['user']} | RISK SCORE: {row['risk_score']}")
    print('='*50)
    print(generate_threat_report(row))

In [ ]:
import json

top10 = profile.sort_values('risk_score', ascending=False).head(10)

results = []

for _, row in top10.iterrows():
    report = generate_threat_report(row)
    results.append({
        'user': row['user'],
        'risk_score': row['risk_score'],
        'report': report
    })
    print(f"Done: {row['user']}")

# Save to file
with open('threat_reports.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\nAll reports saved to threat_reports.json")

In [ ]:
import json
import pandas as pd

# Load reports
with open('threat_reports.json', 'r') as f:
    results = json.load(f)

# Build summary table
summary = pd.DataFrame([{
    'User': r['user'],
    'Risk Score': r['risk_score'],
    'Threat Level': 'CRITICAL' if r['risk_score'] >= 90 else 'HIGH' if r['risk_score'] >= 70 else 'MEDIUM'
} for r in results])

print(summary.to_string(index=False))

In [ ]:
with open('dashboard.html', 'w', encoding='utf-8') as f:
    f.write("""
<!DOCTYPE html>
<html>
<head>
    <title>Insider Threat Dashboard</title>
    <style>
        body { font-family: Arial, sans-serif; background: #0d1117; color: #e6edf3; padding: 20px; }
        h1 { color: #58a6ff; text-align: center; }
        .subtitle { text-align: center; color: #8b949e; margin-bottom: 30px; }
        table { width: 100%; border-collapse: collapse; margin-top: 20px; }
        th { background: #161b22; color: #58a6ff; padding: 12px; text-align: left; }
        td { padding: 12px; border-bottom: 1px solid #21262d; }
        tr:hover { background: #161b22; cursor: pointer; }
        .critical { color: #ff4444; font-weight: bold; }
        .high { color: #ff8c00; font-weight: bold; }
        .medium { color: #ffd700; font-weight: bold; }
        .score-bar { background: #21262d; border-radius: 4px; height: 8px; width: 100px; display: inline-block; }
        .score-fill { height: 8px; border-radius: 4px; }
        .report-box { display:none; background: #161b22; border: 1px solid #30363d; 
                      border-radius: 8px; padding: 20px; margin-top: 10px; white-space: pre-wrap; }
        .stats { display: flex; gap: 20px; margin-bottom: 30px; justify-content: center; }
        .stat-card { background: #161b22; border: 1px solid #30363d; border-radius: 8px; 
                     padding: 20px; text-align: center; min-width: 150px; }
        .stat-number { font-size: 2em; font-weight: bold; color: #58a6ff; }
        .stat-label { color: #8b949e; font-size: 0.9em; }
    </style>
</head>
<body>
    <h1>🛡️ Insider Threat Detection System</h1>
    <p class="subtitle">AI-Powered Behavioural Analysis | CERT r4.2 Dataset</p>

    <div class="stats">
        <div class="stat-card">
            <div class="stat-number" style="color:#ff4444">3</div>
            <div class="stat-label">Critical Users</div>
        </div>
        <div class="stat-card">
            <div class="stat-number" style="color:#ff8c00">3</div>
            <div class="stat-label">High Risk Users</div>
        </div>
        <div class="stat-card">
            <div class="stat-number" style="color:#ffd700">4</div>
            <div class="stat-label">Medium Risk Users</div>
        </div>
        <div class="stat-card">
            <div class="stat-number">1000</div>
            <div class="stat-label">Total Users Monitored</div>
        </div>
    </div>

    <table>
        <tr>
            <th>User</th>
            <th>Risk Score</th>
            <th>Threat Level</th>
            <th>AI Assessment</th>
        </tr>
""")

    for r in results:
        score = r['risk_score']
        user = r['user']
        report = r['report'].replace('\n', '\\n').replace("'", "\\'")
        
        if score >= 90:
            level = "<span class='critical'>● CRITICAL</span>"
            color = "#ff4444"
        elif score >= 70:
            level = "<span class='high'>● HIGH</span>"
            color = "#ff8c00"
        else:
            level = "<span class='medium'>● MEDIUM</span>"
            color = "#ffd700"

        f.write(f"""
        <tr onclick="toggle('{user}')">
            <td>{user}</td>
            <td>
                <div class='score-bar'>
                    <div class='score-fill' style='width:{score}px; background:{color}'></div>
                </div>
                {score}
            </td>
            <td>{level}</td>
            <td>Click to expand ▼</td>
        </tr>
        <tr>
            <td colspan='4'>
                <div id='{user}' class='report-box'>{r['report']}</div>
            </td>
        </tr>
""")

    f.write("""
    </table>
    <script>
        function toggle(id) {
            var el = document.getElementById(id);
            el.style.display = el.style.display === 'block' ? 'none' : 'block';
        }
    </script>
</body>
</html>
""")

print("Dashboard saved to dashboard.html")

In [ ]:
import pandas as pd

insiders = pd.read_csv('E:/Project/answers/answers/insiders.csv')

print(insiders.columns.tolist())
print(insiders.head())

In [ ]:
r42_insiders = insiders[insiders['dataset'] == 4.2]

print(r42_insiders[['dataset', 'scenario', 'user']].to_string(index=False))
print(f"\nTotal malicious users in r4.2: {r42_insiders['user'].nunique()}")

In [ ]:
# List of actual malicious users
actual_malicious = set(r42_insiders['user'].tolist())

# Our top 10 flagged users
our_flagged = set(profile.sort_values('risk_score', ascending=False).head(10)['user'].tolist())

# Check overlap
caught = our_flagged.intersection(actual_malicious)
missed = actual_malicious - our_flagged

print(f"Our top 10 flagged users: {our_flagged}")
print(f"\nActually malicious users we caught: {caught}")
print(f"Number caught: {len(caught)} out of 10 flagged")
print(f"\nTotal malicious users missed: {len(missed)} out of 70")

In [ ]:
# Profile the actual malicious users
malicious_list = r42_insiders['user'].tolist()

malicious_profile = profile[profile['user'].isin(malicious_list)]

print("Average behaviour of ACTUAL malicious users:")
print(malicious_profile[['total_logons', 'after_hours_logons', 'usb_events', 'file_events']].describe())

In [ ]:
import pandas as pd
import numpy as np

logon_df = pd.read_csv('E:/Project/r4.2/logon.csv')
logon_df['date'] = pd.to_datetime(logon_df['date'])
logon_df['hour'] = logon_df['date'].dt.hour
logon_df['day'] = logon_df['date'].dt.date

# Daily logon counts per user
daily = logon_df.groupby(['user', 'day']).size().reset_index(name='daily_logons')

# Each user's personal baseline
baseline = daily.groupby('user')['daily_logons'].agg(['mean', 'std']).reset_index()
baseline.columns = ['user', 'mean_daily', 'std_daily']
baseline['std_daily'] = baseline['std_daily'].fillna(1)

# After hours per user
after_hours = logon_df[
    (logon_df['activity'] == 'Logon') &
    ((logon_df['hour'] < 7) | (logon_df['hour'] >= 19))
].groupby('user').size().reset_index(name='after_hours')

# Merge
new_profile = baseline.merge(after_hours, on='user', how='left').fillna(0)

# Z-score based risk — how abnormal is each user relative to themselves
new_profile['after_hours_ratio'] = new_profile['after_hours'] / (new_profile['mean_daily'] + 1)
new_profile['volatility'] = new_profile['std_daily'] / (new_profile['mean_daily'] + 1)

# New risk score
new_profile['risk_score'] = (
    new_profile['after_hours_ratio'] * 0.6 +
    new_profile['volatility'] * 0.4
)

# Normalise
new_profile['risk_score'] = (
    (new_profile['risk_score'] - new_profile['risk_score'].min()) /
    (new_profile['risk_score'].max() - new_profile['risk_score'].min()) * 100
).round(2)

# Check how many malicious users are in top 70
actual_malicious = set(r42_insiders['user'].tolist())
top70 = set(new_profile.sort_values('risk_score', ascending=False).head(70)['user'].tolist())
caught = top70.intersection(actual_malicious)

print(f"Malicious users caught in top 70: {len(caught)} out of 70")
print(caught)

In [ ]:
device_df = pd.read_csv('E:/Project/r4.2/device.csv')
file_df = pd.read_csv('E:/Project/r4.2/file.csv', usecols=['user'])

# USB events per user
usb = device_df.groupby('user').size().reset_index(name='usb_events')

# File events per user
files = file_df.groupby('user').size().reset_index(name='file_events')

# Merge into new profile
new_profile = new_profile.merge(usb, on='user', how='left')
new_profile = new_profile.merge(files, on='user', how='left')
new_profile = new_profile.fillna(0)

# USB and file ratios relative to logon baseline
new_profile['usb_ratio'] = new_profile['usb_events'] / (new_profile['mean_daily'] + 1)
new_profile['file_ratio'] = new_profile['file_events'] / (new_profile['mean_daily'] + 1)

# Rebuild risk score with all signals
new_profile['risk_score'] = (
    new_profile['after_hours_ratio'] * 0.3 +
    new_profile['volatility'] * 0.2 +
    new_profile['usb_ratio'] * 0.3 +
    new_profile['file_ratio'] * 0.2
)

# Normalise
new_profile['risk_score'] = (
    (new_profile['risk_score'] - new_profile['risk_score'].min()) /
    (new_profile['risk_score'].max() - new_profile['risk_score'].min()) * 100
).round(2)

# Recheck
top70 = set(new_profile.sort_values('risk_score', ascending=False).head(70)['user'].tolist())
caught = top70.intersection(actual_malicious)

print(f"Malicious users caught in top 70: {len(caught)} out of 70")
print(caught)

In [ ]:
# Rebuild the best model (12 catches)
new_profile['risk_score'] = (
    new_profile['after_hours_ratio'] * 0.6 +
    new_profile['volatility'] * 0.4
)

# Normalise
new_profile['risk_score'] = (
    (new_profile['risk_score'] - new_profile['risk_score'].min()) /
    (new_profile['risk_score'].max() - new_profile['risk_score'].min()) * 100
).round(2)

# Verify
top70 = set(new_profile.sort_values('risk_score', ascending=False).head(70)['user'].tolist())
caught = top70.intersection(actual_malicious)
print(f"Malicious users caught in top 70: {len(caught)} out of 70")

In [ ]:
# USB deviation from personal baseline
usb_daily = device_df.groupby(['user', device_df.columns[1]]).size().reset_index(name='daily_usb')
usb_baseline = usb_daily.groupby('user')['daily_usb'].agg(['mean', 'std']).reset_index()
usb_baseline.columns = ['user', 'usb_mean', 'usb_std']
usb_baseline['usb_std'] = usb_baseline['usb_std'].fillna(1)
usb_baseline['usb_volatility'] = usb_baseline['usb_std'] / (usb_baseline['usb_mean'] + 1)

# File deviation from personal baseline
file_df2 = pd.read_csv('E:/Project/r4.2/file.csv', usecols=['user', 'date'])
file_df2['date'] = pd.to_datetime(file_df2['date']).dt.date
file_daily = file_df2.groupby(['user', 'date']).size().reset_index(name='daily_files')
file_baseline = file_daily.groupby('user')['daily_files'].agg(['mean', 'std']).reset_index()
file_baseline.columns = ['user', 'file_mean', 'file_std']
file_baseline['file_std'] = file_baseline['file_std'].fillna(1)
file_baseline['file_volatility'] = file_baseline['file_std'] / (file_baseline['file_mean'] + 1)

# Merge into profile
new_profile = new_profile.merge(usb_baseline[['user', 'usb_volatility']], on='user', how='left')
new_profile = new_profile.merge(file_baseline[['user', 'file_volatility']], on='user', how='left')
new_profile = new_profile.fillna(0)

# New risk score with deviation-based USB and file
new_profile['risk_score'] = (
    new_profile['after_hours_ratio'] * 0.4 +
    new_profile['volatility'] * 0.2 +
    new_profile['usb_volatility'] * 0.2 +
    new_profile['file_volatility'] * 0.2
)

# Normalise
new_profile['risk_score'] = (
    (new_profile['risk_score'] - new_profile['risk_score'].min()) /
    (new_profile['risk_score'].max() - new_profile['risk_score'].min()) * 100
).round(2)

# Check
top70 = set(new_profile.sort_values('risk_score', ascending=False).head(70)['user'].tolist())
caught = top70.intersection(actual_malicious)
print(f"Malicious users caught in top 70: {len(caught)} out of 70")
print(caught)

In [ ]:
import pandas as pd
import numpy as np

# Read HTTP in chunks
http_daily_counts = {}

for chunk in pd.read_csv('E:/Project/r4.2/http.csv', usecols=['user', 'date'], chunksize=50000):
    chunk['date'] = pd.to_datetime(chunk['date']).dt.date
    grouped = chunk.groupby(['user', 'date']).size()
    for (user, date), count in grouped.items():
        if user not in http_daily_counts:
            http_daily_counts[user] = {}
        http_daily_counts[user][date] = http_daily_counts[user].get(date, 0) + count

# Build baseline
http_rows = []
for user, days in http_daily_counts.items():
    vals = list(days.values())
    mean = np.mean(vals)
    std = np.std(vals) if len(vals) > 1 else 1
    http_rows.append({'user': user, 'http_volatility': std / (mean + 1)})

http_baseline = pd.DataFrame(http_rows)
print(f"HTTP baseline built for {len(http_baseline)} users")